In [ ]:
## IMPORTS AND SETUP
# Jupyter Notebook setup
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
sys.path.insert(0, "/tf/projet") # Add the project root directory to the Python path (docker hosting)

# Imports
import wandb
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tensorflow.keras.models import Model

from Leyanda_Project.utils.naming import generate_captioning_model_name
from Leyanda_Project.utils.warning_clean import silence_tensorflow_warnings
from Leyanda_Project.models.callbacks import create_callbacks, BLEUCallback
from Leyanda_Project.preprocessing.captioning_preprocessing import load_coco_dataset, create_tokenizer, create_dataset_generator, split_dataset
from Leyanda_Project.models.captioning_models import create_image_encoder_basic, create_image_encoder_attention, create_gru_caption_decoder, create_lstm_caption_decoder_basic, create_lstm_caption_decoder_with_attention, create_captioning_model_basic, create_captioning_model_attention, basic_loss, regularized_semantic_loss, plot_training_history

# Suppress warnings
silence_tensorflow_warnings()

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU is available: {len(gpus)} device(s) detected")
    except RuntimeError as e:
        print("Error configuring GPU:", str(e))
else:
    print("No GPU available, using CPU")

# Wandb setup
if not os.path.exists("/tf/projet/.env"):
    print("WARNING: No .env file found, please create one with your Wandb API key.")
    exit(1)
else:
    load_dotenv("/tf/projet/.env")
    WANDB_API_KEY = os.getenv("API_KEY")
    wandb_entity = "tom-antoine-cesi"

In [ ]:
## PARAMETERS
seed = 123
project_name = "Leyanda"
np.random.seed(seed)
tf.random.set_seed(seed)

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset_livrable_3"  # Path to the raw data folder
images_folder = os.path.join(raw_data_path, "train2017")  # Path to the images folder
annotations_folder = os.path.join(raw_data_path, "annotations")  # Path to the annotations folder
max_samples = 20000 # Maximum number of samples to load for testing
shuffle = False # Shuffle the dataset or not
batch_size = 128  # Batch size for dataset loading
max_length = 25  # Maximum caption length
vocab_size_limit = 20000  # Maximum vocabulary size
train_split = 0.8  # Proportion of the dataset to use for training
val_split = 0.1  # Proportion of the dataset to use for validation
image_size = 180 # Image size (square)

# Model parameters
use_gru = False # False = LSTM, True = GRU
learning_rate = 0.001 # Learning rate
embedding_dim = 512  # Dimension of word embeddings
units = 512  # Number of units in LSTM layers
epochs = 15  # Number of epochs for training
dropout_rate = 0.3 # Dropout rate for LSTM layers
save_path = "/tf/projet/Leyanda_Project/models/saved/L3" # Path to the models folder
semantic_loss = True # True to use semantic loss
encoder_fine_tune_layers = 20 # Number of encoder layers to fine tune, can be 0 or any number of layers
attention = False # True to use LSTM spatial attention (WIP)

# Callbacks parameters
use_wandb = True, # Whether to use wandb for logging
early_stopping = True, # Whether to use early stopping
model_checkpoint = True # Whether to save the best model
BLEU = False # Whether to use BLEU score for evaluation (takes time)
reduce_lr_on_plateau = True # Whether to use adjusted learning rate system

# Inference parameters
num_examples = 10 # Number of inference samples

In [ ]:
## DATA PREPARATION WORKFLOW
print(f"Starting data preparation workflow for {project_name}...")

# Load COCO dataset
image_paths, captions = load_coco_dataset(
    images_folder=images_folder,
    annotations_folder=annotations_folder,
    annotation_file="captions_train2017.json"
)

# Limit dataset size for testing (remove for full training)
if len(image_paths) > max_samples:
    print(f"Limiting dataset to {max_samples} samples for testing")
    random_indices = np.random.choice(len(image_paths), max_samples, replace=False)
    image_paths = [image_paths[i] for i in random_indices]
    captions = [captions[i] for i in random_indices]

# Create and fit tokenizer
tokenizer, vocab_size = create_tokenizer(captions, num_words=vocab_size_limit)

# Split dataset
(train_img_paths, train_captions,
 val_img_paths, val_captions,
 test_img_paths, test_captions) = split_dataset(
    image_paths,
    captions,
    train_split=train_split,
    val_split=val_split
)

# Create TensorFlow datasets
print("Creating TensorFlow datasets...")
train_dataset = create_dataset_generator(
    train_img_paths,
    train_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size,
    shuffle=shuffle
)

val_dataset = create_dataset_generator(
    val_img_paths,
    val_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size,
    shuffle=shuffle
)

test_dataset = create_dataset_generator(
    test_img_paths,
    test_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size,
    shuffle=shuffle
)

for (images, captions_input), captions_target in train_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Captions input shape: {captions_input.shape}")
    print(f"Captions target shape: {captions_target.shape}")
    break

print("Data preparation complete!")

In [ ]:
## MODELS CREATION WORKFLOW
print(f"Starting model creation workflow for {project_name}...")

# Generate model name using parameters
model_name = generate_captioning_model_name(
    project_name=project_name,
    is_gru=use_gru,
    attention=attention,
    semantic_loss=semantic_loss,
    reduce_lr_on_plateau=reduce_lr_on_plateau
)

if use_wandb:
    # Initialize wandb
    run_wandb = wandb.init(
        project=project_name,
        name=model_name,
        entity=wandb_entity,
        config={
            "architecture": "InceptionV3-LSTM",
            "dataset": "COCO",
            "embedding_dim": embedding_dim,
            "units": units,
            "max_length": max_length,
            "vocab_size": len(tokenizer.word_index) + 1,
            "batch_size": batch_size
        }
    )

# Create model
print(f"Creating model {model_name}...")
vocab_size = len(tokenizer.word_index) + 1
if use_gru:
    encoder = create_image_encoder_basic(
        input_shape=(180, 180, 3),
        embedding_dim=embedding_dim,
        fine_tune_layers=encoder_fine_tune_layers
    )
    decoder = create_gru_caption_decoder(
        vocab_size=vocab_size,
        max_length=max_length,
        embedding_dim=embedding_dim,
        units=units,
        dropout_rate=dropout_rate
    )
    captioning_model = create_captioning_model_basic(
        encoder=encoder,
        decoder=decoder,
        max_length=max_length,
        image_size=image_size
    )
else:
    if attention:
        encoder = create_image_encoder_attention(
            input_shape=(image_size, image_size, 3),
            embedding_dim=embedding_dim,
            fine_tune_layers=encoder_fine_tune_layers
        )
        decoder = create_lstm_caption_decoder_with_attention(
            vocab_size=vocab_size,
            max_length=max_length,
            embedding_dim=embedding_dim,
            units=units,
            dropout_rate=dropout_rate
        )
        captioning_model = create_captioning_model_attention(
            encoder=encoder,
            decoder=decoder,
            max_length=max_length,
            image_size=image_size
        )
    else:
        encoder = create_image_encoder_basic(
            input_shape=(image_size, image_size, 3),
            embedding_dim=embedding_dim,
            fine_tune_layers=encoder_fine_tune_layers
        )
        decoder = create_lstm_caption_decoder_basic(
            vocab_size=vocab_size,
            max_length=max_length,
            embedding_dim=embedding_dim,
            units=units,
            dropout_rate=dropout_rate
        )
        captioning_model = create_captioning_model_basic(
            encoder=encoder,
            decoder=decoder,
            max_length=max_length,
            image_size=image_size
        )


# Compile the model
print("Compiling the model...")
if semantic_loss:
    loss_func = regularized_semantic_loss
else:
    loss_func = basic_loss

captioning_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss=loss_func,
    metrics=['accuracy']
)

# Display model summary
print("Model architecture summary:")
captioning_model.summary()

print("Models created successfully!")

In [ ]:
## MODELS TRAINING WORKFLOW
print(f"Starting model training workflow for {project_name}...")

# Create callbacks
callbacks = create_callbacks(
    model_name=f"{project_name}_captioning",
    tensorboard=wandb,
    early_stopping=early_stopping,
    model_checkpoint=model_checkpoint,
)
if use_wandb:
    wandb_callback = wandb.keras.WandbMetricsLogger()
    callbacks.append(wandb_callback)
if BLEU:
    bleu_callback = BLEUCallback(
        val_img_paths[:100],
        val_captions[:100],
        tokenizer,
        max_length,
        save_path
    )
    callbacks.append(bleu_callback)
if reduce_lr_on_plateau:
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    callbacks.append(reduce_lr)


# Train the model
print(f"Training the model for {epochs} epochs...")
history = captioning_model.fit(
    train_dataset,
    epochs=epochs,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=2
)
plot_training_history(history)

# Evaluate the model on validation dataset
print("Evaluating the model on test dataset...")
results = captioning_model.evaluate(test_dataset)
print(f"Test Loss: {results[0]:.4f}")
print(f"Test Accuracy: {results[1]:.4f}")

# Save the models
print("Saving models...")
os.makedirs(save_path, exist_ok=True)
captioning_model.save(os.path.join(f"{save_path}/full", f"full_{model_name}.keras"))
encoder.save(os.path.join(f"{save_path}/encoder", f"encoder_{model_name}.keras"))
decoder.save(os.path.join(f"{save_path}/decoder", f"decoder_{model_name}.keras"))
print(f"Models saved to {save_path}")

if use_wandb:
    wandb.log({
        "final_test_loss": results[0],
        "final_test_accuracy": results[1]
    })
    run_wandb.finish()

print("Model training complete!")

In [ ]:
## INFERENCE WORKFLOW
print(f"Starting inference workflow for {project_name}...")

test_indices = np.random.choice(len(test_img_paths), num_examples, replace=False)

plt.figure(figsize=(15, 25))

for i, idx in enumerate(test_indices):
    img_path = test_img_paths[idx]
    actual_caption = test_captions[idx]
    predicted_caption = captioning_model.predict(img_path)
    plt.subplot(num_examples, 1, i+1)
    img = plt.imread(img_path)
    plt.imshow(img)
    plt.title(f'Actual: {actual_caption}\nPredicted: {predicted_caption}', fontsize=12)
    plt.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(save_path, f"{project_name}_examples.png"))
plt.show()

print("Inference complete!")